In [1]:
from time import time
import pandas as pd
import numpy as np
from collections import OrderedDict
import warnings

import pandas as pd
from CBFV.composition import generate_features

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.ensemble import (
    AdaBoostClassifier, GradientBoostingClassifier,
    RandomForestClassifier, ExtraTreesClassifier
)

from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC, LinearSVC

from sklearn.preprocessing import StandardScaler

In [2]:
def instantiate_model(model_name):
    model = model_name()
    return model

def fit_model(model, X_train, y_train):
    ti = time()
    model = instantiate_model(model)
    model.fit(X_train, y_train)
    fit_time = time() - ti
    return model, fit_time

def append_result_df(df, result_dict):
    df_result_appended = pd.concat([df, pd.DataFrame([result_dict])], ignore_index=True)
    return df_result_appended

def append_model_dict(dic, model_name, model):
    dic[model_name] = model
    return dic

def evaluate_model(model, X, y_act):
    y_pred = model.predict(X)
    acc = accuracy_score(y_act, y_pred)
    f1 = f1_score(y_act, y_pred, average="weighted", zero_division=0)
    precision = precision_score(y_act, y_pred, average="weighted", zero_division=0)
    recall = recall_score(y_act, y_pred, average="weighted", zero_division=0)
    return acc, f1, precision, recall

def fit_evaluate_model(model, model_name, X_train, y_train, X_val, y_val):
    model, fit_time = fit_model(model, X_train, y_train)
    acc_train, f1_train, prec_train, rec_train = evaluate_model(model, X_train, y_train)
    acc_val, f1_val, prec_val, rec_val = evaluate_model(model, X_val, y_val)
    result_dict = {
        'model_name': model_name,
        'model_name_pretty': type(model).__name__,
        'model_params': model.get_params(),
        'fit_time': fit_time,
        'acc_train': acc_train,
        'f1_train': f1_train,
        'precision_train': prec_train,
        'recall_train': rec_train,
        'acc_val': acc_val,
        'f1_val': f1_val,
        'precision_val': prec_val,
        'recall_val': rec_val,
    }
    return model, result_dict

# Reading in dataset

In [3]:
df_train = pd.read_csv('control_dataset_splits/ICSD_train_split.csv')
df_test = pd.read_csv('control_dataset_splits/ICSD_test_split.csv')
df_val = pd.read_csv('control_dataset_splits/ICSD_val_split.csv')

In [4]:
df_train["Crystal_System"] = df_train["Crystal_System"].astype("category")
df_test["Crystal_System"] = df_test["Crystal_System"].astype("category")
df_val["Crystal_System"] = df_val["Crystal_System"].astype("category")


df_train["target"] = df_train["Crystal_System"].cat.codes
df_test["target"] = df_test["Crystal_System"].cat.codes
df_val["target"] = df_val["Crystal_System"].cat.codes

label_mapping = dict(enumerate(df_train["Crystal_System"].cat.categories))

In [5]:
rename_dict = {'Chemical': 'formula'}

df_train = df_train.rename(columns=rename_dict)
df_val = df_val.rename(columns=rename_dict)
df_test = df_test.rename(columns=rename_dict)

In [6]:
# Separate features from target BEFORE passing to CBFV
feature_cols = [col for col in df_train.columns if col not in ['HMS', 'Crystal_System', 'target']]

# CBFV needs 'formula' and 'target' columns only
df_train_cbfv = df_train[['formula', 'target']].copy()
df_val_cbfv   = df_val[['formula', 'target']].copy()
df_test_cbfv  = df_test[['formula', 'target']].copy()

# Generate Oliynyk  features

In [7]:
X_train_unscaled, y_train, formulae_train, skipped_train = generate_features(
    df_train_cbfv, elem_prop='oliynyk', drop_duplicates=False, extend_features=True, sum_feat=True)
X_val_unscaled, y_val, formulae_val, skipped_val = generate_features(
    df_val_cbfv, elem_prop='oliynyk', drop_duplicates=False, extend_features=True, sum_feat=True)
X_test_unscaled, y_test, formulae_test, skipped_test = generate_features(
    df_test_cbfv, elem_prop='oliynyk', drop_duplicates=False, extend_features=True, sum_feat=True)

Processing Input Data: 100%|██████████| 1617/1617 [00:00<00:00, 19360.98it/s]


	Featurizing Compositions...


Assigning Features...: 100%|██████████| 1617/1617 [00:00<00:00, 16626.84it/s]


	Creating Pandas Objects...


Processing Input Data: 100%|██████████| 438/438 [00:00<00:00, 22854.68it/s]


...

	Creating Pandas Objects...


Processing Input Data: 100%|██████████| 228/228 [00:00<00:00, 12374.66it/s]


	Featurizing Compositions...


Assigning Features...: 100%|██████████| 228/228 [00:00<00:00, 13732.66it/s]


	Creating Pandas Objects...


In [8]:
X_train_unscaled

,sum_Atomic_Number,sum_Atomic_Weight,sum_Period,sum_group,sum_families,sum_Metal,sum_Nonmetal,sum_Metalliod,sum_Mendeleev_Number,sum_l_quantum_number,...,mode_polarizability(A^3),mode_Melting_point_(K),mode_Boiling_Point_(K),mode_Density_(g/mL),mode_specific_heat_(J/g_K)_,mode_heat_of_fusion_(kJ/mol)_,mode_heat_of_vaporization_(kJ/mol)_,mode_thermal_conductivity_(W/(m_K))_,mode_heat_atomization(kJ/mol),mode_Cohesive_energy
0,210.60,479.548780,30.90,107.20,51.15,4.0,5.45,0.0,609.15,12.45,...,0.793,54.75,90.15,0.00143,0.92,0.22259,3.4099,0.02674,249.0,2.62
1,208.36,475.068948,30.34,102.72,49.19,4.0,5.17,0.0,584.79,12.17,...,0.793,54.75,90.15,0.00143,0.92,0.22259,3.4099,0.02674,249.0,2.62
2,137.20,287.384750,28.00,113.20,56.00,4.0,6.00,0.0,667.60,12.00,...,0.793,54.75,90.15,0.00143,0.92,0.22259,3.4099,0.02674,249.0,2.62
3,137.60,288.609350,28.00,113.60,56.00,4.0,6.00,0.0,668.80,12.00,...,0.793,54.75,90.15,0.00143,0.92,0.22259,3.4099,0.02674,249.0,2.62
4,137.80,289.221650,28.00,113.80,56.00,4.0,6.00,0.0,669.40,12.00,...,0.793,54.75,90.15,0.00143,0.92,0.22259,3.4099,0.02674,249.0,2.62
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1612,119.30,256.795925,20.40,90.10,40.00,3.0,4.00,0.0,520.50,9.20,...,0.793,54.75,90.15,0.00143,0.92,0.22259,3.4099,0.02674,249.0,2.62
1613,119.20,256.705030,20.40,90.00,40.00,3.0,4.00,0.0,520.20,9.20,...,0.793,54.75,90.15,0.00143,0.92,0.22259,3.4099,0.02674,249.0,2.62
1614,111.45,235.746714,20.00,89.45,39.99,3.0,4.00,0.0,518.06,10.00,...,0.793,54.75,90.15,0.00143,0.92,0.22259,3.4099,0.02674,249.0,2.62
1615,111.35,235.528892,20.00,89.35,39.97,3.0,4.00,0.0,517.18,10.00,...,0.793,54.75,90.15,0.00143,0.92,0.22259,3.4099,0.02674,249.0,2.62


In [9]:
print(f"Skipped train: {len(skipped_train)}")
print(f"Skipped val:   {len(skipped_val)}")
print(f"Skipped test:  {len(skipped_test)}")

Skipped train: 0
Skipped val:   0
Skipped test:  0


In [10]:
print(feature_cols)
print(X_train_unscaled.shape)  # should be (n_samples, n_features)
print(y_train.shape)  # should be (n_samples,)
print(label_mapping)

['formula', 'Temperature', 'Pressure']
(1617, 308)
(1617,)
{0: 'cubic', 1: 'hexagonal', 2: 'monoclinic', 3: 'orthorhombic', 4: 'tetragonal', 5: 'triclinic', 6: 'trigonal'}


In [11]:
from sklearn.preprocessing import normalize
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_unscaled)  # fit only on train
X_val   = scaler.transform(X_val_unscaled)
X_test  = scaler.transform(X_test_unscaled)

# X_train = normalize(X_train)
# X_val = normalize(X_val)
# X_test = normalize(X_test)

In [12]:
df_classics = pd.DataFrame(columns=[
    'model_name',
    'model_name_pretty',
    'model_params',
    'fit_time',
    'acc_train',
    'f1_train',
    'precision_train',
    'recall_train',
    'acc_val',
    'f1_val',
    'precision_val',
    'recall_val'
])

RANDOM_SEED = 42

classic_model_classes = OrderedDict({
    'dumc': DummyClassifier,
    'lr':   LogisticRegression,
    'abc':  AdaBoostClassifier,
    'gbc':  GradientBoostingClassifier,
    'rfc':  RandomForestClassifier,
    'etc':  ExtraTreesClassifier,
    'svc':  SVC,
    'lsvc': LinearSVC,
    'knc':  KNeighborsClassifier,
})

# Models with seeds and convergence fixes applied
classic_model_names = OrderedDict({
    'dumc': lambda: DummyClassifier(random_state=RANDOM_SEED),
    'lr':   lambda: LogisticRegression(random_state=RANDOM_SEED, max_iter=1000),
    'abc':  lambda: AdaBoostClassifier(random_state=RANDOM_SEED),
    'gbc':  lambda: GradientBoostingClassifier(random_state=RANDOM_SEED),
    'rfc':  lambda: RandomForestClassifier(random_state=RANDOM_SEED),
    'etc':  lambda: ExtraTreesClassifier(random_state=RANDOM_SEED),
    'svc':  lambda: SVC(random_state=RANDOM_SEED),
    'lsvc': lambda: LinearSVC(random_state=RANDOM_SEED, max_iter=5000),
    'knc':  lambda: KNeighborsClassifier(),  # no random_state needed
})

df_classics = pd.DataFrame()
classic_models = OrderedDict()

ti = time()
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for model_name, model in classic_model_names.items():
        print(f'Now fitting and evaluating model {model_name}: {model}')
        model, result_dict = fit_evaluate_model(
            model, model_name, X_train, y_train, X_val, y_val
        )
        df_classics = append_result_df(df_classics, result_dict)
        classic_models = append_model_dict(classic_models, model_name, model)

dt = time() - ti
print(f'Finished fitting {len(classic_models)} models, total time: {dt:0.2f} s')

Now fitting and evaluating model dumc: <function <lambda> at 0x000001B878A82660>
Now fitting and evaluating model lr: <function <lambda> at 0x000001B82B5F3380>
Now fitting and evaluating model abc: <function <lambda> at 0x000001B82B5F3100>
Now fitting and evaluating model gbc: <function <lambda> at 0x000001B82B5F1EE0>
Now fitting and evaluating model rfc: <function <lambda> at 0x000001B82B5F31A0>
Now fitting and evaluating model etc: <function <lambda> at 0x000001B82B5F1260>
Now fitting and evaluating model svc: <function <lambda> at 0x000001B82B5F3560>
Now fitting and evaluating model lsvc: <function <lambda> at 0x000001B82B5F0900>
Now fitting and evaluating model knc: <function <lambda> at 0x000001B82B5F3600>
Finished fitting 9 models, total time: 105.28 s


In [13]:
# Sort in order of increasing validation r2 score
df_classics = df_classics.sort_values('acc_val', ignore_index=True)
df_classics

,model_name,model_name_pretty,model_params,fit_time,acc_train,f1_train,precision_train,recall_train,acc_val,f1_val,precision_val,recall_val
0,dumc,DummyClassifier,"{'constant': None, 'random_state': 42, 'strate...",0.000000,0.313544,0.149686,0.098310,0.313544,0.273973,0.117838,0.075061,0.273973
1,abc,AdaBoostClassifier,"{'algorithm': 'deprecated', 'estimator': None,...",1.859658,0.594310,0.575872,0.582430,0.594310,0.534247,0.501536,0.518546,0.534247
2,lr,LogisticRegression,"{'C': 1.0, 'class_weight': None, 'dual': False...",1.451853,0.806432,0.804744,0.805577,0.806432,0.694064,0.686561,0.687565,0.694064
3,svc,SVC,"{'C': 1.0, 'break_ties': False, 'cache_size': ...",0.226009,0.789116,0.786543,0.789954,0.789116,0.698630,0.687988,0.693761,0.698630
4,lsvc,LinearSVC,"{'C': 1.0, 'class_weight': None, 'dual': 'auto...",32.448358,0.828077,0.825967,0.828412,0.828077,0.700913,0.694817,0.702676,0.700913
5,knc,KNeighborsClassifier,"{'algorithm': 'auto', 'leaf_size': 30, 'metric...",0.000000,0.821892,0.821120,0.822446,0.821892,0.739726,0.736555,0.736083,0.739726
6,gbc,GradientBoostingClassifier,"{'ccp_alpha': 0.0, 'criterion': 'friedman_mse'...",63.891897,0.972171,0.972173,0.972447,0.972171,0.796804,0.797636,0.804572,0.796804
7,etc,ExtraTreesClassifier,"{'bootstrap': False, 'ccp_alpha': 0.0, 'class_...",0.577831,0.985776,0.985760,0.985989,0.985776,0.828767,0.828728,0.830437,0.828767
8,rfc,RandomForestClassifier,"{'bootstrap': True, 'ccp_alpha': 0.0, 'class_w...",1.368814,0.985776,0.985767,0.985869,0.985776,0.828767,0.828809,0.830403,0.828767


In [14]:
from tqdm import tqdm
# Combine train and val for final training
X_train_final = np.concatenate((X_train, X_val), axis=0)
y_train_final = np.concatenate((y_train, y_val), axis=0)

seeds = [42, 123, 456, 789, 1024, 2024, 314, 99, 7, 2000]
models_to_evaluate = ['etc', 'rfc', 'lr', 'lsvc']
seed_rows = []

for seed in tqdm(seeds, desc='Seeds'):
    for model_name in tqdm(models_to_evaluate, desc='Models', leave=False):
        
        model_params = df_classics.loc[df_classics['model_name'] == model_name].iloc[0]['model_params'].copy()
        model_params['random_state'] = seed
        
        model = classic_model_classes[model_name](**model_params)
        model.fit(X_train_final, y_train_final)
        
        acc, f1, precision, recall = evaluate_model(model, X_test, y_test)
        
        seed_rows.append({
            'seed':      seed,
            'model':     model_name,
            'test_acc':  acc,
            'test_f1':   f1,
            'test_prec': precision,
            'test_rec':  recall,
        })

seed_df = pd.DataFrame(seed_rows)

summary = seed_df.groupby('model').agg(
    acc_mean=('test_acc', 'mean'),
    acc_std= ('test_acc', 'std'),
    f1_mean= ('test_f1',  'mean'),
    f1_std=  ('test_f1',  'std'),
).round(4)

print(summary)

seed_df['feature_set'] = 'oliynyk' 
seed_df.to_csv('results/oliynyk_seed_results.csv', index=False)

Seeds: 100%|██████████| 10/10 [10:07<00:00, 60.72s/it]

       acc_mean  acc_std  f1_mean  f1_std
model                                    
etc      0.8592   0.0053   0.8608  0.0049
lr       0.6798   0.0000   0.6761  0.0000
lsvc     0.7193   0.0000   0.7168  0.0000
rfc      0.8566   0.0069   0.8588  0.0063
